In [ ]:
!pip install sentencepiece
!pip install git+https://github.com/brandnewchoppa/rope-tensorflow.git

In [ ]:
import pandas as pd
import tensorflow as tf
import sentencepiece as spm
import json
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
from rope_tensorflow import RoPE

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(resolver)
    # Initialize the TPU before creating distributed resources.
    tf.tpu.experimental.initialize_tpu_system(resolver)
    print("All devices: ", tf.config.list_logical_devices('TPU'))
    strategy = tf.distribute.TPUStrategy(resolver)
except:
    strategy = tf.distribute.get_strategy()
    print("⚠️ Running on CPU/GPU")

In [ ]:
class Sequence_Tokenizer:
    def __init__(self, vocab_path, k=6):
        self.vocab_path = vocab_path
        self.k = k
        with open(self.vocab_path, "r") as file:
            self.vocab = json.load(file)

    def tokenize_sequence(self, seq):
        tokens = self.kmer_tokenize(seq)
        t = [self.vocab.get(tok, self.vocab.get("<unk>", 3)) for tok in tokens]

        t.insert(0, 1)
        t.append(2)
        return t

    def kmer_tokenize(self, sequence):
        return [sequence[i:i+self.k] for i in range(len(sequence) - self.k + 1)]

    def get_vocab_size(self):
        return len(self.vocab)

In [ ]:
sequence_tokenizer = Sequence_Tokenizer("/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/vocab_protein.json")
text_sp = spm.SentencePieceProcessor(model_file='/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/input_effect_text.model')
changes_sp = spm.SentencePieceProcessor(model_file='/kaggle/input/dna-mutations-tokenizers/tensorflow2/default/4/tokenizers/changes.model')

In [ ]:
def merge_ds(ds_human, parsed_human, output_path='fixed.csv'):

    ds_human = pd.read_csv(ds_human)
    parsed_human = pd.read_csv(parsed_human)

    merged_df = pd.merge(ds_human, parsed_human, how='inner', on='effect')

    if 'Effect' in merged_df.columns:
        effect_col = 'Effect'
    elif 'effect' in merged_df.columns:
        effect_col = 'effect'
    else:
        raise ValueError("No 'effect' or 'Effect' column found in merged dataset")

    mask = ~(
        (~merged_df[effect_col].str.contains(';', na=False)) &
        (merged_df['changes'].str.contains('{', na=False)) |
        ((merged_df['normal_sequence'].str.len())>2056)
    )
    merged_df = merged_df[mask].copy()

    print(len(merged_df))

    return merged_df

In [ ]:
merged_df = merge_ds('/kaggle/input/full-unlinked-mutations-ds/merged.csv', '/kaggle/input/full-unlinked-mutations-ds/parsed_full.csv') 

In [ ]:
def make_ds(df, sequence_tokenizer, text_sp, changes_sp, batch_size=32,
            max_input_len_dna=2056, max_input_len_text=128, max_output_len=32):
    df = df.dropna(subset=["normal_sequence", "parsed_effects", "changes"])

    full_length = 0

    for d in df["normal_sequence"]:
        if len(d)>full_length:
            full_length=len(d)

    df["input_sequence"] = df["normal_sequence"].apply(sequence_tokenizer.tokenize_sequence)
    df["input_text"] = df["parsed_effects"].apply(
        lambda x: text_sp.encode(str(x), out_type=int, add_bos=True, add_eos=True)
    )
    df["output_changes"] = df["changes"].apply(
        lambda x: changes_sp.encode(str(x), out_type=int, add_bos=True, add_eos=True)
    )

    df["input_sequence"] = list(
        pad_sequences(df["input_sequence"], maxlen=max_input_len_dna, padding="post", truncating="post")
    )
    df["input_text"] = list(
        pad_sequences(df["input_text"], maxlen=max_input_len_text, padding="post", truncating="post")
    )
    df["output_changes"] = list(
        pad_sequences(df["output_changes"], maxlen=max_output_len, padding="post", truncating="post")
    )

    input_sequences = np.stack(df["input_sequence"].to_numpy())
    input_texts = np.stack(df["input_text"].to_numpy())
    output_changes = np.stack(df["output_changes"].to_numpy())

    decoder_input = output_changes[:, :-1]   # teacher forcing input
    decoder_target = output_changes[:, 1:]   # target prediction

    dataset = tf.data.Dataset.from_tensor_slices(
        ((input_sequences, input_texts, decoder_input), decoder_target)
    )

    dataset = dataset.shuffle(buffer_size=len(df), seed=42, reshuffle_each_iteration=False)

    train_size = int(0.7 * len(df))
    val_size = int(0.2 * len(df))
    test_size = len(df) - train_size - val_size

    train_ds = dataset.take(train_size)
    val_ds = dataset.skip(train_size).take(val_size)
    test_ds = dataset.skip(train_size + val_size)

    train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    print(f"✅ Dataset prepared: {len(df)} samples "
          f"({train_size} train, {val_size} val, {test_size} test)")

    return train_ds, val_ds, test_ds


In [ ]:
train_ds, val_ds, test_ds = make_ds(merged_df, sequence_tokenizer, text_sp, changes_sp)

In [ ]:
for (input_sequences, input_texts, decoder_input), decoder_target in val_ds:
    break

print(input_sequences)

In [ ]:
class PositionalEmbeddingEncoder(tf.keras.layers.Layer):
  def __init__(self, max_len, vocab_size_dna, vocab_size_text, d_model):
    super().__init__()
    self.d_model = d_model

    self.token_emb_dna = tf.keras.layers.Embedding(vocab_size_dna, d_model)
    self.token_emb_text = tf.keras.layers.Embedding(vocab_size_text, d_model)

    self.pos_emb = RoPE(dim=d_model)

    # Segment embeddings (DNA=0, Text=1)
    self.seg_emb = tf.keras.layers.Embedding(2, d_model)

  def compute_mask(self, context_dna, context_text, **kwargs):
    mask_dna = self.token_emb_dna.compute_mask(context_dna)
    mask_text = self.token_emb_text.compute_mask(context_text)
    if mask_dna is not None and mask_text is not None:
        return tf.concat([mask_dna, mask_text], axis=1)
    return None

  def call(self, context_dna, context_text):
    batch_size = tf.shape(context_dna)[0]
    L1 = tf.shape(context_dna)[1]
    L2 = tf.shape(context_text)[1]

    dna_emb = self.token_emb_dna(context_dna)   # (B, L1, d_model)
    text_emb = self.token_emb_text(context_text) # (B, L2, d_model)

    concat_emb = tf.concat([dna_emb, text_emb], axis=1)  # (B, L1+L2, d_model)

    pos_emb = self.pos_emb.rotate(concat_emb)  # (L1+L2, d_model)

    dna_seg = tf.zeros((batch_size, L1), dtype=tf.int32)  # DNA=0
    text_seg = tf.ones((batch_size, L2), dtype=tf.int32)  # Text=1
    seg_ids = tf.concat([dna_seg, text_seg], axis=1)      # (B, L1+L2)
    seg_emb = self.seg_emb(seg_ids)                       # (B, L1+L2, d_model)

    x = concat_emb + pos_emb + seg_emb
    return x

In [ ]:
class PositionalEmbeddingDecoder(tf.keras.layers.Layer):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.d_model = d_model
    self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, mask_zero=True)
    self.pos_emb = RoPE(dim=d_model)

  def compute_mask(self, *args, **kwargs):
    return self.embedding.compute_mask(*args, **kwargs)

  def call(self, x):
    x = self.embedding(x)
    y = self.pos_emb.rotate(x)
    x = x+y
    return x

In [ ]:
class BaseAttention(tf.keras.layers.Layer):
  def __init__(self, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.GroupQueryAttention(**kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

In [ ]:
class CrossAttention(BaseAttention):
  def call(self, x, context):
    attn_output, attn_scores = self.mha(
        query=x,
        key=context,
        value=context,
        return_attention_scores=True)

    # Retain attention scores for plotting.
    self.last_attn_scores = attn_scores

    x = self.add([x, attn_output])
    x = self.layernorm(x)

    return x

In [ ]:
class GlobalSelfAttention(BaseAttention):
  def call(self, x):
    attn_output = self.mha(
        query=x,
        value=x,
        key=x)
    x = self.add([x, attn_output])
    x = self.layernorm(x)
    return x

In [ ]:
class CausalSelfAttention(BaseAttention):
  def call(self, x):
    attn_output = self.mha(
        query=x,
        value=x,
        key=x,
        use_causal_mask = True)
    x = self.add([x, attn_output])
    x = self.layernorm(x)
    return x

In [ ]:
class FeedForward(tf.keras.layers.Layer):
  def __init__(self, d_model, dff, dropout_rate=0.1):
    super().__init__()
    self.seq = tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='relu'),
      tf.keras.layers.Dense(d_model),
      tf.keras.layers.Dropout(dropout_rate)
    ])
    self.add = tf.keras.layers.Add()
    self.layer_norm = tf.keras.layers.LayerNormalization()

  def call(self, x):
    x = self.add([x, self.seq(x)])
    x = self.layer_norm(x)
    return x

In [ ]:
class EncoderLayer(tf.keras.layers.Layer):
  def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()

    self.self_attention = GlobalSelfAttention(
        num_query_heads=num_heads,
        num_key_value_heads=4,
        head_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x):
    x = self.self_attention(x)
    x = self.ffn(x)
    return x

In [ ]:
class DNA_CNN_Downsampler(tf.keras.layers.Layer):
    def __init__(
        self,
        d_model,
        kernel_sizes=[15, 11, 7],
        strides=[2, 2, 2],
        activation='gelu',
        dropout=0.1,
        use_batch_norm=True,
        **kwargs
    ):
        super().__init__(**kwargs)

        filters = [d_model//4, d_model//2, d_model]

        self.filters = filters
        self.kernel_sizes = kernel_sizes
        self.strides = strides
        self.activation = activation
        self.dropout_rate = dropout
        self.use_batch_norm = use_batch_norm

        self.conv_blocks = []
        for i, (f, k, s) in enumerate(zip(filters, kernel_sizes, strides)):
            conv_block = []

            conv_block.append(tf.keras.layers.Conv1D(
                filters=f,
                kernel_size=k,
                strides=s,
                padding='same',
                activation=None,
                name=f'conv_{i}'
            ))

            if use_batch_norm:
                conv_block.append(tf.keras.layers.BatchNormalization(name=f'bn_{i}'))

            conv_block.append(tf.keras.layers.Activation(activation, name=f'act_{i}'))

            if dropout > 0:
                conv_block.append(tf.keras.layers.Dropout(dropout, name=f'dropout_{i}'))

            self.conv_blocks.append(conv_block)

        self.total_downsampling = 1
        for s in strides:
            self.total_downsampling *= s

    def call(self, x, training=None):
        for conv_block in self.conv_blocks:
            for layer in conv_block:
                if isinstance(layer, (tf.keras.layers.Dropout, tf.keras.layers.BatchNormalization)):
                    x = layer(x, training=training)
                else:
                    x = layer(x)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'filters': self.filters,
            'kernel_sizes': self.kernel_sizes,
            'strides': self.strides,
            'activation': self.activation,
            'dropout_rate': self.dropout_rate,
            'use_batch_norm': self.use_batch_norm
        })
        return config

In [ ]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads,
                 dff, vocab_size_text, vocab_size_dna, max_len=2048, dropout_rate=0.1):
        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.cnn = DNA_CNN_Downsampler(d_model=d_model, dropout=dropout_rate)

        self.embedding = PositionalEmbeddingEncoder(max_len=max_len,
                                                    d_model=d_model,
                                                    vocab_size_dna=vocab_size_dna,
                                                    vocab_size_text=vocab_size_text)

        self.enc_layers = [
            EncoderLayer(d_model=d_model,
                         num_heads=num_heads,
                         dff=dff,
                         dropout_rate=dropout_rate)
            for _ in range(num_layers)
        ]
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, context_dna, context_text):
        x = self.embedding(context_dna, context_text)
        x = self.cnn(x)
        x = self.dropout(x)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x)

        return x  # (B, L1+L2, d_model)

In [ ]:
class DecoderLayer(tf.keras.layers.Layer):
  def __init__(self,
               *,
               d_model,
               num_heads,
               dff,
               dropout_rate=0.1):
    super(DecoderLayer, self).__init__()

    self.causal_self_attention = CausalSelfAttention(
        num_query_heads=num_heads,
        num_key_value_heads=4,
        head_dim=d_model,
        dropout=dropout_rate)

    self.cross_attention = CrossAttention(
        num_query_heads=num_heads,
        num_key_value_heads=4,
        head_dim=d_model,
        dropout=dropout_rate)

    self.ffn = FeedForward(d_model, dff)

  def call(self, x, context):
    x = self.causal_self_attention(x=x)
    x = self.cross_attention(x=x, context=context)

    # Retain the last attention scores for plotting.
    self.last_attn_scores = self.cross_attention.last_attn_scores

    x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
    return x

In [ ]:
class Decoder(tf.keras.layers.Layer):
  def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size,
               dropout_rate=0.1):
    super(Decoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.pos_embedding = PositionalEmbeddingDecoder(vocab_size=vocab_size,
                                             d_model=d_model)
    self.dropout = tf.keras.layers.Dropout(dropout_rate)
    self.dec_layers = [
        DecoderLayer(d_model=d_model, num_heads=num_heads,
                     dff=dff, dropout_rate=dropout_rate)
        for _ in range(num_layers)]

    self.last_attn_scores = None

  def call(self, x, context):
    # `x` is token-IDs shape (batch, target_seq_len)
    x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)

    x = self.dropout(x)

    for i in range(self.num_layers):
      x  = self.dec_layers[i](x, context)

    self.last_attn_scores = self.dec_layers[-1].last_attn_scores

    # The shape of x is (batch_size, target_seq_len, d_model).
    return x

In [ ]:
class Transformer(tf.keras.Model):
  def __init__(self, *, num_layers, d_model, num_heads, dff,
               input_vocab_size_dna, input_vocab_size_text,
               target_vocab_size, dropout_rate=0.1, max_len=6000):
    super().__init__()

    self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           vocab_size_dna=input_vocab_size_dna,
                           vocab_size_text=input_vocab_size_text,
                           dropout_rate=dropout_rate, max_len=max_len)

    self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                           num_heads=num_heads, dff=dff,
                           vocab_size=target_vocab_size,
                           dropout_rate=dropout_rate)

    self.final_layer = tf.keras.layers.Dense(target_vocab_size)

  def call(self, inputs):
    # Keras fit expects all inputs in the first argument.
    context_dna, context_text, x  = inputs

    context = self.encoder(context_dna, context_text)  # (batch_size, context_len, d_model)

    x = self.decoder(x, context)  # (batch_size, target_len, d_model)

    logits = self.final_layer(x)  # (batch_size, target_len, target_vocab_size)

    try:
      # Remove the Keras mask to avoid rescaling losses and metrics.
      # b/250038731
      del logits._keras_mask
    except AttributeError:
      pass

    return logits

In [ ]:
num_layers = 6
d_model = 512
dff = 2048
num_heads = 8
dropout_rate = 0.1

In [ ]:
with strategy.scope():
    transformer = Transformer(
        num_layers=num_layers,
        d_model=d_model,
        num_heads=num_heads,
        dff=dff,
        max_len=2056,
        input_vocab_size_dna=sequence_tokenizer.get_vocab_size(),
        input_vocab_size_text=text_sp.get_piece_size(),
        target_vocab_size=changes_sp.get_piece_size(),
        dropout_rate=dropout_rate)

In [ ]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
  def __init__(self, d_model, warmup_steps=20000):
    super().__init__()

    self.d_model = d_model
    self.d_model = tf.cast(self.d_model, tf.float32)

    self.warmup_steps = warmup_steps

  def __call__(self, step):
    step = tf.cast(step, dtype=tf.float32)
    arg1 = tf.math.rsqrt(step)
    arg2 = step * (self.warmup_steps ** -1.5)

    return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

In [ ]:
with strategy.scope():
    learning_rate = CustomSchedule(d_model)

    optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98,
                                         epsilon=1e-9)

In [ ]:
def masked_loss(label, pred):
  pred = tf.clip_by_value(pred, -10.0, 10.0)

  mask = label != 0
  loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')
  loss = loss_object(label, pred)

  mask = tf.cast(mask, dtype=loss.dtype)
  loss *= mask

  loss = tf.math.divide_no_nan(tf.reduce_sum(loss), tf.reduce_sum(mask))
  return loss

def masked_accuracy(label, pred):
  pred = tf.argmax(pred, axis=2)
  label = tf.cast(label, pred.dtype)
  match = label == pred

  mask = label != 0

  match = match & mask

  match = tf.cast(match, dtype=tf.float32)
  mask = tf.cast(mask, dtype=tf.float32)
  return tf.math.divide_no_nan(tf.reduce_sum(match), tf.reduce_sum(mask))

In [ ]:
class MaskedExactMatch(tf.keras.metrics.Metric):
    def __init__(self, name='masked_exact_match', **kwargs):
        super(MaskedExactMatch, self).__init__(name=name, **kwargs)
        self.total_matches = self.add_weight(name='total_matches', initializer='zeros')
        self.total_valid = self.add_weight(name='total_valid', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        """
        Update the metric state.
        
        Args:
            y_true: Ground truth labels (shape: [batch_size, ...])
            y_pred: Predicted labels or logits (shape: [batch_size, ...])
            sample_weight: Optional weighting (not implemented in this version)
        """
        if len(y_pred.shape) > len(y_true.shape):
            y_pred = tf.argmax(y_pred, axis=-1)

        y_true = tf.cast(y_true, tf.int64)
        y_pred = tf.cast(y_pred, tf.int64)

        # Exclude padding tokens (ID 0).
        mask = tf.not_equal(y_true, 0)

        matches = tf.equal(y_true, y_pred)

        # Count matches only at non-padding positions.
        masked_matches = tf.logical_and(matches, mask)

        self.total_matches.assign_add(tf.reduce_sum(tf.cast(masked_matches, tf.float32)))
        self.total_valid.assign_add(tf.reduce_sum(tf.cast(mask, tf.float32)))

    def result(self):
        """Return the current metric value."""
        return tf.math.divide_no_nan(self.total_matches, self.total_valid)

    def reset_state(self):
        """Reset the metric state."""
        self.total_matches.assign(0.0)
        self.total_valid.assign(0.0)

In [ ]:
with strategy.scope():
    transformer.compile(
        loss=masked_loss,
        optimizer=optimizer,
        metrics=[masked_accuracy, MaskedExactMatch()])

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('checkpoints/best_model.weights.h5', monitor='val_loss', save_best_only=True, save_weights_only=True),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.TensorBoard(update_freq=1000, histogram_freq=1),
]

In [ ]:
transformer.fit(train_ds,
                epochs=3,
                validation_data=val_ds,
                callbacks=callbacks,
                verbose=2)

In [ ]:
result = transformer.evaluate(test_ds, verbose=2)
print(result)

In [ ]:
class DnaMutator(tf.Module):
    def __init__(self, tokenizers, transformer):
        self.tokenizers = tokenizers
        self.transformer = transformer

    def __call__(self, dna_sequence, input_text, max_length=512):
        if isinstance(dna_sequence, str) and isinstance(input_text, str):
            seq_tokens = self.tokenizers['dna_sequence'].tokenize_sequence(dna_sequence)
            encoder_sequence = tf.convert_to_tensor([seq_tokens], dtype=tf.int32)  # shape (1, seq_len)

            input_text = self.tokenizers['input_text'].encode(input_text, out_type=int, add_bos=True, add_eos=True)
            encoder_text = tf.convert_to_tensor([input_text], dtype=tf.int32)  # shape (1, seq_len)

        else:
            raise ValueError("sequence must be a string")

        BOS_ID = 1
        EOS_ID = 2

        output_array = tf.TensorArray(dtype=tf.int32, size=0, dynamic_size=True)
        output_array = output_array.write(0, BOS_ID)

        for i in tf.range(max_length):
            decoder_input = tf.transpose(output_array.stack())[tf.newaxis, :]  # shape (1, seq_len)

            predictions = self.transformer([encoder_sequence, encoder_text, decoder_input], training=False)
            predictions = predictions[:, -1, :]  # shape (1, vocab_size)
            predicted_id = tf.argmax(predictions, axis=-1, output_type=tf.int32)

            output_array = output_array.write(i+1, predicted_id[0])

            if predicted_id[0] == EOS_ID:
                break

        output_seq = tf.transpose(output_array.stack())[tf.newaxis, :]  # shape (1, tokens)

        tokens_list = output_seq.numpy().tolist()[0]
        text = self.tokenizers['changes'].decode(tokens_list)

        return text, tokens_list

In [ ]:
tokenizers = {
    'dna_sequence': sequence_tokenizer,
    'input_text':text_sp,
    'changes': changes_sp
}

Mutator = DnaMutator(tokenizers, transformer)

dna_seq = "AGAACGCGTGTGGGCGGCCCGGGCGGCGTCTGGCCCTCTCCCCATCCTCGCGCGCCGCGCCCCAGCCCCGGGATGCTGCTCCGGCCCCGCAGGCCGCCCCCGCTCGCGCCCCCCGCGCCGCCCTCGCCCGCCAGCCCCGACCCCGAGCCGCGGACACCCGGAGACGCCCCGGGGACCCCGCCCCGGAGGCCCGCCTCGCCCAGCGCGCTGGGGGAACTCGGGTTGCCGGTGTCCCCGGGCTCGGCGCAGCGCACGCCCTGGAGCGCCCGGGAGACGGAGCTGCTGCTGGGGACGCTGCTGCAACCGGCCGTGTGGCGCGCGCTGCTCCTGGACCGCCGCCAGGCCCTGCCCACCTACCGCCGCGTGTCGGCCGCGCTGGCCCAGCAGCAGGTGCGCCGCACCCCCGCGCAGTGCCGCCGCCGCTACAAGTTCCTTAAAGACAAGTTTCGCGAGGCGCACGGCCAGCCGCCCGGGCCCTTCGACGAGCAGATCCGGAAGCTCATGGGGCTGCTGGGCGACAACGGGCGCAAACGGCCTCGCCGCCGCTCCCCGGGGTCCGGGCGCCCCCAGCGCGCCCGCCGCCCGGTCCCCAACGCGCACGCGCCGGCTCCCAGCGAACCAGACGCCACCCCGCTGCCCACCGCCCGCGACCGCGACGCGGACCCCACCTGGACGCTCCGCTTCAGCCCGTCCCCACCGAAGTCTGCGGACGCCTCCCCCGCCCCCGGCTCCCCGCCAGCTCCCGCCCCGACCGCCCTCGCCACCTGCATCCCCGAGGACCGCGCGCCCGTCCGCGGCCCCGGGTCCCCGCCGCCACCCCCGGCCCGCGAAGACCCCGACTCGCCGCCCGGCCGCCCCGAGGACTGCGCGCCCCCTCCGGCCGCGCCCCCGTCGCTGAACACCGCCCTGCTGCAGACCCTGGGGCACCTGGGCGACATCGCGAACATCCTGGGCCCGCTGCGCGACCAGCTGCTGACCTTGAACCAGCACGTGGAGCAGCTGCGCGGCGCCTTCGACCAGACAGTGTCCCTGGCCGTGGGCTTCATTCTGGGCAGCGCGGCCGCCGAGCGAGGGGTCCTCAGGGACCCGTGCCAGTGAGTCCCGGCTGCGGCCAGTCTCCCCTCTCCAGGCCGAGGCCCCTCCGCCCTGACCCCTCCTGCTGCCTTCCCACCCCCGTTCTTGGGTATGTTCAATAAAAGGATTGTTTTCCTAGA" #human insulin P01308
text_change = 'Make this sequence abolish coactivation activity.'

text, tokens = Mutator(dna_seq, text_change)
print("Generated text:", text)
print("Token IDs:", tokens)